In [1]:
#!pip install --upgrade comtradeapicall

In [2]:
import pandas as pd
import requests
import comtradeapicall
import numpy as np
from itertools import count
import json
import math
import os
import requests
from IPython.display import clear_output

In [3]:
## configuration
relative_data_path = r'C:\Users\Administrator\Personal-Projects\Trade_data_Analysis\Data'
subscription_key = os.getenv("comtrade_subscription_key")
pd.set_option('display.max_columns', None)

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
# READ H0 HS DATA
url = "https://comtradeapi.un.org/files/v1/app/reference/H6.json"
response = requests.get(url)
data = response.json()
HS_2022 = pd.DataFrame(data)
HS_2022.head()

,more,minimumInputLength,classCode,className,results
0,False,2,H6,HS2022,"{'id': 'TOTAL', 'text': 'Total - All H6 commod..."
1,False,2,H6,HS2022,"{'id': '01', 'text': '01 - Animals; live', 'pa..."
2,False,2,H6,HS2022,"{'id': '0101', 'text': '0101 - Horses, asses, ..."
3,False,2,H6,HS2022,"{'id': '010121', 'text': '010121 - Horses; liv..."
4,False,2,H6,HS2022,"{'id': '010129', 'text': '010129 - Horses; liv..."


In [6]:
f = open('C:\\Users\\Administrator\\Personal-Projects\\Trade_data_Analysis\\Trade notebooks\\HS.json',encoding='utf-8')
hs_data = json.load(f)

In [7]:
path2 = 'C:\\Users\\Administrator\\Personal-Projects\\Trade_data_Analysis\\Trade notebooks\\country_codes_V202401b.csv'
country_codes = pd.read_csv(path2)
codes = country_codes['country_code'].to_list()
print(codes)
country_codes.head()

[4, 8, 12, 16, 20, 24, 28, 31, 32, 36, 40, 44, 48, 50, 51, 52, 56, 58, 60, 64, 68, 70, 72, 76, 84, 86, 90, 92, 96, 100, 104, 108, 112, 116, 120, 124, 132, 136, 140, 144, 148, 152, 156, 162, 166, 170, 174, 175, 178, 180, 184, 188, 191, 192, 196, 200, 203, 204, 208, 212, 214, 218, 222, 226, 231, 232, 233, 238, 242, 246, 251, 258, 260, 262, 266, 268, 270, 275, 276, 278, 280, 288, 292, 296, 300, 304, 308, 316, 320, 324, 328, 332, 340, 344, 348, 352, 360, 364, 368, 372, 376, 380, 384, 388, 392, 398, 400, 404, 408, 410, 414, 417, 418, 422, 426, 428, 430, 434, 440, 442, 446, 450, 454, 458, 462, 466, 470, 478, 480, 484, 490, 496, 498, 499, 500, 504, 508, 512, 516, 520, 524, 528, 530, 531, 533, 534, 535, 540, 548, 554, 558, 562, 566, 570, 574, 579, 580, 583, 584, 585, 586, 591, 598, 600, 604, 608, 612, 616, 620, 624, 626, 634, 642, 643, 646, 652, 654, 659, 660, 662, 666, 670, 674, 678, 682, 686, 688, 690, 694, 697, 699, 702, 703, 704, 705, 706, 710, 711, 716, 724, 728, 729, 736, 740, 748, 752, 

,country_code,country_name,country_iso2,country_iso3
0,4,Afghanistan,AF,AFG
1,8,Albania,AL,ALB
2,12,Algeria,DZ,DZA
3,16,American Samoa,AS,ASM
4,20,Andorra,AD,AND


### Fuctions to Extract the Data

In [8]:
def getFinalDataBulkAvailability(subscription_key, typeCode, freqCode, clCode, period, reporterCode, publishedDateFrom=None, publishedDateTo=None):
    return getDataAvailability(subscription_key, 'FINAL', 'BULK', typeCode, freqCode, clCode, period, reporterCode, publishedDateFrom, publishedDateTo)


In [9]:
def split_numbers(input_string, chunk_size):
    # Split the string by commas to get a list of numbers
    numbers = input_string.split(',')

    # Group the numbers into chunks of specified size
    chunks = [numbers[i:i + chunk_size] for i in range(0, len(numbers), chunk_size)]

    return chunks

In [10]:
def get_data_from_uncomtrade(reporter,partner,flowCode,frequency,data_date,hs_code,hs_2_digit=None,hs_4_digit=None,hs_6_digit=None,include_none=False):

    """
    Fetch trade data from UN Comtrade based on specified criteria.

    Parameters:
    - reporter (str or list): Reporting country or countries.
    - partner (str or list): Partner country or countries.
    - frequency (str): Data frequency (e.g., 'annual', 'monthly').
    - data_date (str): Data period(s) in a comma-separated string.
    - hs_code (list or str): HS codes to retrieve data for, or "00" for all 2-digit codes.
    - hs_2_digit, hs_4_digit, hs_6_digit (bool): Flags to include specific HS code levels.
    - include_none (bool): If False, excludes rows with None values in the output.

    Returns:
    - final_dataframe (DataFrame): Raw trade data from the API.
    - test_dataframe (DataFrame): Processed trade data with comparisons.
    """

    if isinstance(data_date, str):
        years_list = data_date.split(',')
        # Check the length of the list and truncate if necessary
        if len(years_list) > 12:
            print("Maximum 12 date periods can be used in date string per call")
            return None,None

    if isinstance(reporter, list):
        string_list = [str(item) for item in reporter]
        reporter = ','.join(map(str, reporter))
    elif isinstance(reporter, str):
        pass
    else:
        print("The reporter codes should be a list or comma seperated string.")
        return None

    if isinstance(partner, list):
        string_list = [str(item) for item in partner]
        partner = ','.join(map(str, partner))
    elif isinstance(partner, str):
        pass
    else:
        print("The partner codes should be a list or comma seperated string.")
        return None

    if isinstance(hs_code, list) or hs_code=="00":
        pass
    else:
        print(f"""HS code must be passed as a list of strings or should be "00" """)
        return None
    if not hs_code:
        result = True
    else:
        # Get the length of the first element
        first_length = len(hs_code[0])

        # Check if all elements have the same length
        result = all(len(item) == first_length for item in hs_code)

    if result==False:
        print("The hs code string should include only 2,4 or 6 digitis as list do not send mixed digits hs code.")
        return None
    ############################################################
    if hs_code=="00":
        hs_list_2_digit=[]
        for item in hs_data["results"]:
            if item["aggrLevel"]==2:
                hs_list_2_digit.append(item['id'])
        codes_string = ','.join(map(str, hs_list_2_digit))
        final_dataframe = comtradeapicall.getFinalData(subscription_key, typeCode='C', freqCode=frequency, clCode='HS', period=data_date,
                                        reporterCode=reporter, cmdCode=codes_string, flowCode=flowCode, partnerCode=partner,
                                        partner2Code=None, customsCode=None, motCode=None, maxRecords=250000,
                                        format_output='JSON', aggregateBy=None, breakdownMode='classic',
                                        countOnly=None, includeDesc=True)
    ############################################################
    else:
        string_list = [str(item) for item in hs_code]
        if len(string_list[0])==2:

            codes_string_2_digit = ','.join(map(str, hs_code))
            hs_list_4_digit=[]
            for item in hs_data["results"]:
                if item["aggrLevel"]==4 and item["parent"] in hs_code:
                    hs_list_4_digit.append(item['id'])
            codes_string_4_digit = ','.join(map(str, hs_list_4_digit))
            hs_list_6_digit=[]
            for item in hs_data["results"]:
                if item["aggrLevel"]==6 and item["parent"] in hs_list_4_digit:
                    hs_list_6_digit.append(item['id'])
            codes_string_6_digit = ','.join(map(str, hs_list_6_digit))


        elif len(string_list[0])==4:
            for item in hs_code:
                unique_first_two_digits = set()
                # Extract the first two digits
                first_two_digits = item[:2]
                # Add the first two digits to the set
                unique_first_two_digits.add(first_two_digits)
            # Convert the set to a list if needed
            unique_first_two_digits_list = list(unique_first_two_digits)
            codes_string_2_digit = ','.join(map(str, unique_first_two_digits_list))
            codes_string_4_digit=','.join(map(str, hs_code))
            hs_list_6_digit=[]
            for item in hs_data["results"]:
                if item["aggrLevel"]==6 and item["parent"] in hs_code:
                    hs_list_6_digit.append(item['id'])
            codes_string_6_digit = ','.join(map(str, hs_list_6_digit))

        elif len(string_list[0])==6:
            for item in hs_code:
                unique_first_four_digits = set()
                # Extract the first two digits
                first_four_digits = item[:4]
                # Add the first two digits to the set
                unique_first_four_digits.add(first_four_digits)
            # Convert the set to a list if needed
            unique_first_four_digits_list = list(unique_first_four_digits)
            codes_string_4_digit = ','.join(map(str, unique_first_four_digits_list))
            for item in unique_first_four_digits_list:
                unique_first_two_digits = set()
                # Extract the first two digits
                first_two_digits = item[:2]
                # Add the first two digits to the set
                unique_first_two_digits.add(first_two_digits)
            # Convert the set to a list if needed
            unique_first_two_digits_list = list(unique_first_two_digits)
            codes_string_2_digit = ','.join(map(str, unique_first_two_digits_list))
            codes_string_6_digit=','.join(map(str, hs_code))

        else:
            print("The hs code string should include only 2,4 or 6 digitis as list do not send mixed digits hs code.")
            return None
        if(hs_2_digit==False and hs_4_digit==False and hs_6_digit==False):
            print("At least one of the digit selections must be True")
            return None
        codes_string=""
        if(hs_2_digit==True):
            codes_string+=codes_string_2_digit+","
        if(hs_4_digit==True):
            codes_string+=codes_string_4_digit+","
        if(hs_6_digit==True):
            codes_string+=codes_string_6_digit
        codes_string = codes_string.rstrip(',')
        print(f"""Final codes_string= {codes_string} """)
        chunk_size = 100
        chunks = split_numbers(codes_string, chunk_size)
        final_dataframe=pd.DataFrame()
        for chunk in range(0,len(chunks),1):
            string_numbers = list(map(str, chunks[chunk]))
            codes_string = ','.join(map(str, string_numbers))
            mydf = comtradeapicall.getFinalData(subscription_key, typeCode='C', freqCode=frequency, clCode='HS', period=data_date,
                                        reporterCode=reporter, cmdCode=codes_string, flowCode=flowCode, partnerCode=partner,
                                        partner2Code=None, customsCode=None, motCode=None, maxRecords=250000,
                                        format_output='JSON', aggregateBy=None, breakdownMode='classic',
                                        countOnly=None, includeDesc=True)
            final_dataframe=pd.concat([final_dataframe,mydf])
        final_dataframe=final_dataframe.reset_index(drop=True)
        return final_dataframe
    ##################################################################### Added for non equal price  check
    columns = ['cifvalue', 'fobvalue', 'primaryValue']
    # Initialize the result to True

    # Initialize an empty DataFrame to store rows with differences
    rows_with_differences = []

    # Loop through each row in the final_dataframe
    for _, row in final_dataframe.iterrows():
        # Extract the values for the specified columns
        values = [row[col] for col in columns]

        # Filter out NaN values and zeros
        non_nan_values = [value for value in values if pd.notna(value) and value != 0.0]

        # Check if all non-NaN values are the same
        if len(set(non_nan_values)) > 1:
            # If there are differences, add the row to the list
            rows_with_differences.append(row)

    # Concatenate all rows with differences into a DataFrame
    failures_dataframe = pd.DataFrame(rows_with_differences)

    # Check if any differences were found and output the result
    if not failures_dataframe.empty:
        print("There are differences in the price columns")

    ####################################################################
    test_dataframe=pd.DataFrame()
    periods_list = final_dataframe['period'].unique().tolist()
    for periods in periods_list:
        filtered_df=final_dataframe[final_dataframe["period"]==periods]

        reporter_numbers = filtered_df['reporterCode'].unique().tolist()
        for reporter_loop in reporter_numbers:
            unique_numbers = filtered_df[filtered_df["reporterCode"]==reporter_loop]['cmdCode'].unique().tolist()
            for hs_codes in unique_numbers:
                mydf=filtered_df[(filtered_df["cmdCode"]==hs_codes)&(filtered_df["reporterCode"]==reporter_loop)]
                unique_partners = mydf['partnerCode'].unique().tolist()
                if len(unique_partners)>1:
                    if unique_partners[0] == 0:
                        unique_partners = unique_partners[1:]

                for partners in unique_partners:
                    mydf=filtered_df[(filtered_df["cmdCode"]==hs_codes)&(filtered_df["reporterCode"]==reporter_loop)&(filtered_df["partnerCode"]==partners)]
                    # Check if mydf is empty
                    if mydf.empty:
                        import_quantity = None
                        export_quantity = None
                        import_amount= None
                        export_amount= None
                        unit_type = None
                        difference = None
                        balance_status = None
                        Hs_description=None
                        country_tag=None
                        reporter=None
                        partner=None
                        difference_amount = None
                        balance_status_dollars = None
                        total_parent=None
                        period_data=None
                    else:
                        flow_list=mydf["flowCode"].unique().tolist()
                        if "X" not in flow_list or "M" not in flow_list:
                            continue

                        try:
                            if mydf[mydf['flowCode'] == 'M']['qty'].values[0]!=0 and mydf[mydf['flowCode'] == 'X']['qty'].values[0]!=0 :
                                import_quantity = mydf[mydf['flowCode'] == 'M']['qty'].values[0]
                                export_quantity = mydf[mydf['flowCode'] == 'X']['qty'].values[0]
                                unit_type = mydf[mydf['flowCode'] == 'X']['qtyUnitAbbr'].values[0] if not mydf[mydf['flowCode'] == 'X']['qtyUnitAbbr'].empty else None
                            else:
                                import_quantity = mydf[mydf['flowCode'] == 'M']['altQty'].values[0] if not mydf[mydf['flowCode'] == 'M']['altQty'].empty else None
                                export_quantity = mydf[mydf['flowCode'] == 'X']['altQty'].values[0] if not mydf[mydf['flowCode'] == 'X']['altQty'].empty else None
                                unit_type = mydf[mydf['flowCode'] == 'X']['altQtyUnitAbbr'].values[0] if not mydf[mydf['flowCode'] == 'X']['altQtyUnitAbbr'].empty else None
                            ################################################ Under this part new value selector selects any not NaN
                            columns = ['cifvalue', 'fobvalue', 'primaryValue']
                            # Initialize variables
                            import_amount = None
                            export_amount = None
                            # Check for the first non-NaN, non-0.0 value for 'M'
                            for col in columns:
                                values = mydf[mydf['flowCode'] == 'M'][col].values
                                if len(values) > 0 and pd.notna(values[0]) and values[0] != 0.0:
                                    import_amount = values[0]
                                    break
                            # Check for the first non-NaN, non-0.0 value for 'X'
                            for col in columns:
                                values = mydf[mydf['flowCode'] == 'X'][col].values
                                if len(values) > 0 and pd.notna(values[0]) and values[0] != 0.0:
                                    export_amount = values[0]
                                    break
                            # If either import_amount or export_amount is None, set both to None
                            if import_amount is None or export_amount is None:
                                import_amount = None
                                export_amount = None

                        except IndexError as e:
                            print("IndexError occurred:", e)
                            print("DataFrame mydf:")
                            return final_dataframe,mydf


                        Hs_description=mydf[mydf['flowCode'] == 'X']['cmdDesc'].values[0] if not mydf[mydf['flowCode'] == 'X']['cmdDesc'].empty else None
                        country_tag = mydf[mydf['flowCode'] == 'X']['reporterISO'].values[0]
                        reporter = mydf[mydf['flowCode'] == 'X']['reporterCode'].values[0]
                        partner_tag= mydf[mydf['flowCode'] == 'X']['partnerDesc'].values[0]
                        partner=mydf[mydf['flowCode'] == 'X']['partnerCode'].values[0]
                        # Calculate the differences and balance status
                        if export_quantity is not None and import_quantity is not None:
                            difference = export_quantity - import_quantity
                            balance_status = "Positive" if difference > 0 else "Negative"
                        else:
                            difference = None
                            balance_status = None

                        if export_amount is not None and import_amount is not None:
                            difference_amount = export_amount - import_amount
                            balance_status_dollars = "Positive" if difference_amount > 0 else "Negative"
                        else:
                            difference_amount = None
                            balance_status_dollars = None


                    if include_none==False:
                        if difference==None and difference_amount==None:
                            continue

                    total_parent=None
                    if len(hs_codes)>2:
                        total_parent=hs_codes[:2]
                    data = pd.DataFrame([{
                    "Period": periods,
                    "HS Code": hs_codes,
                    "Parent": total_parent,
                    "Reporter Code": reporter,
                    "Reporter": country_tag,
                    "Partner Code": partner,
                    "Partner": partner_tag,
                    "HS_Description": Hs_description,
                    "Unit Type": unit_type,
                    "Import Quantitiy": import_quantity,
                    "Export Quantitiy": export_quantity,
                    "Import/Export Diff Quantity": difference,
                    "Import/Export Balance Quantity": balance_status,
                    "Import Amount Dollars": import_amount,
                    "Export Amount Dollars": export_amount,
                    "Import/Export Diff Dollars": difference_amount,
                    "Import/Export Balance Dollars": balance_status_dollars
                }])
                    test_dataframe = pd.concat([test_dataframe, data], ignore_index=True)
    if len(failures_dataframe)>0:
        return final_dataframe,test_dataframe
    else:
        return final_dataframe,test_dataframe

Get all the HS codes data reportered


In [11]:
uncomtrade_data_hs_2,balanced_dataframe_hs_2=get_data_from_uncomtrade(reporter="270",partner=codes, flowCode ="X",frequency="A",data_date="2022",hs_code="00",hs_2_digit=False,hs_4_digit=False,hs_6_digit=True)

In [12]:
positive_top_5_items=uncomtrade_data_hs_2["cmdCode"].unique().tolist()

This calls all the exports/imports data of all countries to a specific country, the values returned are close to the onesin Atlas Dataset and BACI Datasets. All you need to change is the partners code and year

In [13]:
gambia_exports_2022 =get_data_from_uncomtrade(reporter="270",partner=codes,data_date="2022",flowCode='X', frequency="A", hs_code=positive_top_5_items,hs_2_digit=False,hs_4_digit=False,hs_6_digit=True)

Final codes_string= 030110,030111,030119,030191,030192,030193,030194,030195,030199,030211,030212,030213,030214,030219,030221,030222,030223,030224,030229,030231,030232,030233,030234,030235,030236,030239,030240,030241,030242,030243,030244,030245,030246,030247,030249,030250,030251,030252,030253,030254,030255,030256,030259,030261,030262,030263,030264,030265,030266,030267,030268,030269,030270,030271,030272,030273,030274,030279,030281,030282,030283,030284,030285,030289,030290,030291,030292,030299,030310,030311,030312,030313,030314,030319,030321,030322,030323,030324,030325,030326,030329,030331,030332,030333,030334,030339,030341,030342,030343,030344,030345,030346,030349,030350,030351,030352,030353,030354,030355,030356,030357,030359,030360,030361,030362,030363,030364,030365,030366,030367,030368,030369,030371,030372,030373,030374,030375,030376,030377,030378,030379,030380,030381,030382,030383,030384,030389,030390,030391,030392,030399,030410,030411,030412,030419,030420,030421,030422,030429,030431,

In [14]:
gambia_exports_2023 =get_data_from_uncomtrade(reporter="270",partner=codes,data_date="2023",flowCode='X', frequency="A", hs_code=positive_top_5_items,hs_2_digit=False,hs_4_digit=False,hs_6_digit=True)

Final codes_string= 030110,030111,030119,030191,030192,030193,030194,030195,030199,030211,030212,030213,030214,030219,030221,030222,030223,030224,030229,030231,030232,030233,030234,030235,030236,030239,030240,030241,030242,030243,030244,030245,030246,030247,030249,030250,030251,030252,030253,030254,030255,030256,030259,030261,030262,030263,030264,030265,030266,030267,030268,030269,030270,030271,030272,030273,030274,030279,030281,030282,030283,030284,030285,030289,030290,030291,030292,030299,030310,030311,030312,030313,030314,030319,030321,030322,030323,030324,030325,030326,030329,030331,030332,030333,030334,030339,030341,030342,030343,030344,030345,030346,030349,030350,030351,030352,030353,030354,030355,030356,030357,030359,030360,030361,030362,030363,030364,030365,030366,030367,030368,030369,030371,030372,030373,030374,030375,030376,030377,030378,030379,030380,030381,030382,030383,030384,030389,030390,030391,030392,030399,030410,030411,030412,030419,030420,030421,030422,030429,030431,

C:\Users\Administrator\AppData\Local\Temp\ipykernel_4528\443933539.py:156: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_dataframe=pd.concat([final_dataframe,mydf])


In [15]:
gambia_exports_2023

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,flowDesc,partnerCode,partnerISO,partnerDesc,partner2Code,partner2ISO,partner2Desc,classificationCode,classificationSearchCode,isOriginalClassification,cmdCode,cmdDesc,aggrLevel,isLeaf,customsCode,customsDesc,mosCode,motCode,motDesc,qtyUnitCode,qtyUnitAbbr,qty,isQtyEstimated,altQtyUnitCode,altQtyUnitAbbr,altQty,isAltQtyEstimated,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,410,KOR,Rep. of Korea,0,W00,World,H6,HS,True,030319,"Fish; frozen, salmonidae, n.e.c. in item no. 0...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,20800.0,False,8,kg,20800.0,False,20800.0,False,0.0,False,None,13372.160,13372.160,0,True,False
1,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,410,KOR,Rep. of Korea,0,W00,World,H6,HS,True,030223,"Fish; fresh or chilled, sole (Solea spp.), exc...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,333361.0,False,8,kg,333361.0,False,333361.0,False,0.0,False,None,231927.573,231927.573,0,True,False
2,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,410,KOR,Rep. of Korea,0,W00,World,H6,HS,True,030279,"Fish; fresh or chilled, Nile perch (Lates nilo...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,240406.0,False,8,kg,240406.0,False,240406.0,False,0.0,False,None,150286.308,150286.308,0,True,False
3,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,410,KOR,Rep. of Korea,0,W00,World,H6,HS,True,030312,"Fish; frozen, Pacific salmon (Oncorhynchus gor...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,21600.0,False,8,kg,21600.0,False,21600.0,False,0.0,False,None,16702.663,16702.663,0,True,False
4,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,408,PRK,Dem. People's Rep. of Korea,0,W00,World,H6,HS,True,030323,"Fish; frozen, tilapias (Oreochromis spp.), exc...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,24500.0,False,8,kg,24500.0,False,24500.0,False,0.0,False,None,6428.627,6428.627,0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,466,MLI,Mali,0,W00,World,H6,HS,True,940490,Bedding and similar furnishing articles; n.e.c...,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,284.0,False,8,kg,284.0,False,284.0,False,0.0,False,None,2221.928,2221.928,0,True,False
215,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,72,BWA,Botswana,0,W00,World,H6,HS,True,940529,"Luminaires and light fittings; electric, table...",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,110.0,False,8,kg,110.0,False,110.0,False,0.0,False,None,98.057,98.057,0,True,False
216,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,686,SEN,Senegal,0,W00,World,H6,HS,True,940690,"Buildings; prefabricated, not of wood",6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,196000.0,False,8,kg,196000.0,False,196000.0,False,0.0,False,None,56316.759,56316.759,0,True,False
217,C,A,20230101,2023,52,2023,270,GMB,Gambia,X,Export,694,SLE,Sierra Leone,0,W00,World,H6,HS,True,961620,Powder puffs and pads; for the application of ...,6,True,C00,TOTAL CPC,0,0,TOTAL MOT,8,kg,8058.0,False,8,kg,8058.0,False,8058.0,False,0.0,False,None,8326.435,8326.435,0,True,False


In [16]:
# filter out the most important variables
gambia_exports_2022 = gambia_exports_2022[['refYear','reporterDesc','partnerDesc','partnerISO','cmdCode', 'fobvalue']]
gambia_exports_2023 = gambia_exports_2023[['refYear','reporterDesc','partnerDesc','partnerISO','cmdCode', 'fobvalue']]

In [17]:
gambia_exports_2022.head()

,refYear,reporterDesc,partnerDesc,partnerISO,cmdCode,fobvalue
0,2022,Gambia,Belgium,BEL,030279,17560.648
1,2022,Gambia,Brazil,BRA,030354,85733.718
2,2022,Gambia,Cyprus,CYP,030279,21569.353
3,2022,Gambia,Benin,BEN,030355,7930.021
4,2022,Gambia,Italy,ITA,030279,557331.537


In [18]:
total_exports_2022 = gambia_exports_2022['fobvalue'].sum()
print(f"Total export value for the year 2022: {total_exports_2022}")

Total export value for the year 2022: 54553080.888000004


In [19]:
print(gambia_exports_2022['cmdCode'].nunique())
print(gambia_exports_2022['partnerDesc'].nunique())

162
46


In [20]:
gambia_exports_2023.head()

,refYear,reporterDesc,partnerDesc,partnerISO,cmdCode,fobvalue
0,2023,Gambia,Rep. of Korea,KOR,030319,13372.160
1,2023,Gambia,Rep. of Korea,KOR,030223,231927.573
2,2023,Gambia,Rep. of Korea,KOR,030279,150286.308
3,2023,Gambia,Rep. of Korea,KOR,030312,16702.663
4,2023,Gambia,Dem. People's Rep. of Korea,PRK,030323,6428.627


In [21]:
print(gambia_exports_2023['cmdCode'].nunique())
print(gambia_exports_2023['partnerDesc'].nunique())

130
50


In [22]:
total_exports_2023 = gambia_exports_2023['fobvalue'].sum()
print(f"Total export value for the year 2023: {total_exports_2023}")

Total export value for the year 2023: 83355633.70699999


In [23]:
# export index
index_df = pd.read_csv('/content/drive/MyDrive/Trade project data/usa_data/d7f111f0-1329-4004-a6dc-a355f2385ec2_Data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Trade project data/usa_data/d7f111f0-1329-4004-a6dc-a355f2385ec2_Data.csv'

In [ ]:
# refine the dataframe
index_df.rename(columns={'Country Name': 'partnerDesc', 'Country Code': 'partnerISO', '2022 [YR2022]': 'index'}, inplace=True)

In [ ]:
index_df= index_df[['partnerISO', 'index']]

In [ ]:
constant_df = pd.merge(partner_export_2022, index_df, on=['partnerISO'], how='left')

In [ ]:
constant_df['index'] = constant_df['index'].replace('..', '100')
# fill null value with 100
constant_df['index'] = constant_df['index'].fillna(100)

In [ ]:
constant_df['index'] = pd.to_numeric(constant_df['index'], errors='coerce')
constant_df['constant_usd'] = constant_df['fobvalue'] / (constant_df['index']) * 100

In [ ]:
constant_df.head()
print(constant_df.shape)

(220, 5)


In [ ]:
constant_df.isna().sum()

,0
partnerDesc,0
partnerISO,0
fobvalue,0
index,0
constant_usd,0


In [ ]:
# print the null values
print(constant_df[constant_df.isna().any(axis=1)])

                   partnerDesc partnerISO      fobvalue  index  constant_usd
5                     Anguilla        AIA  5.712335e+06    NaN           NaN
26      Br. Indian Ocean Terr.        IOT  3.921630e+05    NaN           NaN
44              Christmas Isds        CXR  8.662690e+05    NaN           NaN
45                  Cocos Isds        CCK  1.342609e+06    NaN           NaN
49                   Cook Isds        COK  9.693480e+05    NaN           NaN
72    Falkland Isds (Malvinas)        FLK  3.135775e+07    NaN           NaN
75   Fr. South Antarctic Terr.        ATF  8.535300e+04    NaN           NaN
130                 Montserrat        MSR  1.485528e+06    NaN           NaN
143                       Niue        NIU  2.627450e+06    NaN           NaN
144               Norfolk Isds        NFK  2.711940e+05    NaN           NaN
148            Other Asia, nes        S19  9.184460e+10    NaN           NaN
156                   Pitcairn        PCN  5.066610e+05    NaN           NaN

In [ ]:
# calculate the total import value
total_imports_2022 = constant_df['constant_usd'].sum()
print(f"Total adjusted import value for the year 2022: {total_imports_2022}")

Total adjusted import value for the year 2022: 2380100036881.688


# services data

In [ ]:
services_df = pd.read_csv('/content/drive/MyDrive/Trade project data/usa_data/Commercial Services 7.31.2024.csv')

In [ ]:
# slice dataframe
usa_services = services_df[services_df['REPORTER CODE'] == 'US']
sorted_usa_services = usa_services[usa_services['FLOW'] == 'Imports'].sort_values('FLOW')

In [ ]:
# sort by specif year
services_df_2022 = sorted_usa_services[sorted_usa_services.YEAR == 2022]

In [ ]:
services_df_2022.head()

,FLOW,INDICATOR_CODE,REPORTER CODE,PARTNER CODE,REPORTER_COUNTRY_CODE,PARTNER_COUNTRY_CODE,REPORTING ECONOMY,PARTNER ECONOMY,SECTOR,YEAR,VALUE,UNIT
42219,Imports,SOX1,US,WL,RC840,SA000,United States of America,World,Other commercial services,2022,406552.0,Million US dollar
34676,Imports,S,US,WL,RC840,SA000,United States of America,World,Memo item: Total services,2022,713886.0,Million US dollar
49747,Imports,SC,US,WL,RC840,SA000,United States of America,World,Transport,2022,157711.0,Million US dollar
26964,Imports,SPX1,US,WL,RC840,SA000,United States of America,World,Memo item: Other services,2022,432099.0,Million US dollar
7287,Imports,SOX,US,WL,RC840,SA000,United States of America,World,Commercial services,2022,688339.0,Million US dollar


From the above dataframe get the Total services from sector column